# Dataset Iris

Referência
- [The Iris Dataset](https://scikit-learn.org/1.4/auto_examples/datasets/plot_iris_dataset.html)

## Setup

Sobre a alocação de GPU, o Keras 3 lê a variável `KERAS_BACKEND` no momento em que o pacote é
importado pela primeira vez.

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "torch")

## Imports

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import keras
import torch

from sklearn import datasets
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Carregar Dataset

In [ ]:
iris = datasets.load_iris()

features = iris.data
labels = iris.target
classes = iris.target_names

## Sobre os Dados

### Visualizar

In [ ]:
pca = PCA(n_components=2)
components = pca.fit_transform(features)

df_pca = pd.DataFrame(components, columns=["PC1", "PC2"])
aux_df = pd.DataFrame(labels, columns=["species"])

df_pca["species"] = aux_df["species"].map(lambda i: classes[i])

sns.scatterplot(df_pca, x="PC1", y="PC2", hue="species")
plt.show()

### Preparar

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

### Normalizar

In [ ]:
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

## Sobre o Modelo

### Montar

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(4,), name='input_layer'),
    keras.layers.Dense(16, activation="relu", name='dense_input_layer'),
    keras.layers.Dense(8, activation="relu", name='dense_hidden_layer'),
    keras.layers.Dense(3, activation="softmax", name='dense_output_layer')
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=['accuracy']
)

model.summary()

### Treinamento

#### Alocar GPU

O treinamento roda na GPU do Apple Silicon por meio do
backend **PyTorch** do Keras 3, que utiliza o dispositivo **MPS**
(Metal Performance Shaders).

A alocação do dispositivo é automática — o backend PyTorch seleciona o
MPS quando disponível.

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Keras backend: {keras.backend.backend()}")
print(f"Training device: {device}")

#### Treinar

In [ ]:
logs = model.fit(
    x_train,
    y_train,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
)

### Avaliar

In [ ]:
lss, acc = model.evaluate(x_test, y_test, verbose=1)

print("lss:", lss)
print("acc:", acc)

### Fazer Predição

In [ ]:
preds = model.predict(x_test)

pred_class = np.argmax(preds, axis=1)
pred_class